In [ ]:
%%capture
%pip install -q -U google-genai
%pip install google-cloud-aiplatform

In [ ]:
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
import pandas as pd
import json5
import time
import os
import dotenv
dotenv.load_dotenv()

## Prompt and paths

In [ ]:
# Cell 3: Configuration
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
MODEL_NAME = "models/gemini-2.5-flash"

input_path = "../data/processed/cleaned_segmented_data.parquet"
output_path = "../data/raw/classified_segmented_data.parquet"

chunk_size = 1500
batch_request_json = "../data/cache/batch_cls_{}.jsonl"

SYSTEM_INSTRUCTION = """
# Vai trò
Bạn là chuyên gia phân tích nội dung.

# Nhiệm vụ
Phân loại văn bản thành định dạng JSON.

- SET is_economic = TRUE nếu nội dung liên quan đến kinh tế.
- SET is_economic = FALSE nếu nội dung không liên quan đến kinh tế.
"""

SYSTEM_INSTRUCTION_OBJECT = {
  "parts": [
    {
      "text": SYSTEM_INSTRUCTION
    }
  ]
}

In [ ]:
client = genai.Client(api_key=GEMINI_API_KEY)

## Create batch data

In [ ]:
# Cell 5: Define output schema and load data
class ClassificationOutput(BaseModel):
    is_economic: bool = Field(description="Phân loại: True nếu nội dung liên quan đến Kinh tế, False nếu không liên quan.")

# Load the input data
data = pd.read_parquet(input_path, engine="pyarrow")
# data = data.sample(n=20, random_state=42)
data.reset_index(drop=True, inplace=True)

# Join chunks into a single text per article
inputs = []
for idx, row in data.iterrows():
    title = row['title']
    url = row['url']
    chunks = row['chunks']  # List[Str]
    
    # Join all chunks into one text for classification
    full_text = "\n\n".join(chunks)
    
    inputs.append({
        'article_index': idx,
        'title': title,
        'url': url,
        'full_text': full_text,
        'chunks': chunks  # Keep original chunks for later
    })

print(f"Total articles for classification: {len(inputs)}")

In [ ]:
# Create JSONL batch files
file_idx = 1
filenames = []

for i in range(0, len(inputs), chunk_size):
    batch = inputs[i : i + chunk_size]

    filename = batch_request_json.format(file_idx)
    file_idx += 1
    
    with open(filename, "w", encoding="utf-8") as f:
        for item in batch:
            idx = item['article_index']
            text = item['full_text']
            record = {
                "key": f"article-{idx}",
                "request": {
                    "contents": [
                        {
                            "parts": [{"text": text}]
                        }
                    ],
                    "system_instruction": SYSTEM_INSTRUCTION_OBJECT,
                    "generation_config": {
                        "max_output_tokens": 200,
                        "temperature": 0.2,
                        "top_p": 0.9,
                        "response_mime_type": "application/json",
                        "response_json_schema": ClassificationOutput.model_json_schema(),
                        "thinking_config": {"thinking_budget": 0}
                    },
                }
            }
            f.write(json5.dumps(record) + "\n")
    
    print("[INFO] Created", filename, "[Articles]", len(batch))
    filenames.append(filename)

## Submit batch data

In [ ]:
# 1. Helper: Upload and Start Batch
def upload_file(filename: str):
    print(f"[START] Uploading {filename}...")
    uploaded = client.files.upload(
        file=filename,
        config=types.UploadFileConfig(
            display_name="classification-requests",
            mime_type="text/plain"
        )
    )
    
    batch_job = client.batches.create(
        model=MODEL_NAME,
        src=uploaded.name,
        config={"display_name": "classification-run"}
    )
    print(f"[INFO] Job started: {batch_job.name}")
    return batch_job

# 2. Helper: Poll for Status (Blocking)
def fetch_job(filename: str, job_name: str):
    print(f"[POLL] Tracking {job_name}...")
    while True:
        job = client.batches.get(name=job_name)
        # Check the state using the enum name string
        state = job.state.name 
        
        # If it's done (Success, Failed, Cancelled), break the loop
        if state not in ("JOB_STATE_RUNNING", "JOB_STATE_PENDING"):
            print(f"[FINISH] {filename} ended with state: {state}")
            break
        
        time.sleep(30) 
    
    if state != "JOB_STATE_SUCCEEDED":
        print(f"[ERROR] Job {job_name} failed. Check dashboard.")
    
    return job

## Submit batch jobs

In [ ]:
# 3. Main Loop: Submit -> Wait -> Repeat
final_results = []

for filename in filenames:
    # A. Submit
    job = upload_file(filename=filename)
    
    # B. Wait for completion (The fix)
    completed_job = fetch_job(filename, job.name)
    
    # C. Store result
    final_results.append((filename, completed_job))

print("\nAll batch jobs processed.")

## Save result

In [ ]:
records_list = []

for filename, job in final_results:
    result_file = job.dest.file_name
    raw = client.files.download(file=result_file).decode('utf-8')
    records = [json5.loads(line) for line in raw.splitlines()]
    records_list.append(records)

In [ ]:
# Cell 10: Process and save results
# Parse classification results
classification_results = {}
errors = []

for records in records_list:
    for rec in records:
        try:
            key = rec.get("key", "")
            article_idx = int(key.split("-")[1]) if key.startswith("article-") else None
            
            result = rec["response"]["candidates"][0]["content"]["parts"][0]["text"]
            structured = json5.loads(result)
            
            classification_results[article_idx] = structured.get("is_economic", False)
        except Exception as e:
            errors.append(rec)

# Create output with one row per article
outputs = []
for item in inputs:
    article_idx = item['article_index']
    is_economic = classification_results.get(article_idx, False)
    
    outputs.append({
        'is_economic': is_economic,
        'title': item['title'],
        'url': item['url'],
        'chunks': item['chunks']  # Keep as List[Str]
    })

df = pd.DataFrame(outputs)
df.to_parquet(output_path, engine="pyarrow", index=False)
print("[INFO] Saved:", output_path)
print(f"[INFO] Total articles: {len(df)}")

In [ ]:
# Cell 11: Handle errors
print(f"[INFO] {len(errors)} failed")

if len(errors) > 0:
    failed_path = "../data/cache/failed_outputs.parquet"
    failed_df = pd.DataFrame({'error_record': [str(e) for e in errors]})
    failed_df.to_parquet(failed_path, engine="pyarrow", index=False)
    
    print("[INFO] Failed records saved to", failed_path)